# Stage 10 -- ICD Codes from Structured Labs and Vitals

**Input** : `structured_labs.json` and `structured_vitals.json` in each admission folder
**Output**: `stage_06e_lab_vital_codes/lab_vital_codes.json`

## Why this stage exists

A third **independent candidate source**, alongside 6b (prior admissions) and 6c (current-visit
diagnoses). It reads nothing from Stages 5/6/6c/6d and can run in any order -- like 6b.

Many ICD-10 codes are assigned directly off a measured value rather than off anything written in
the narrative. Measured on this cohort, **34 of 247 ground-truth codes (13.8%)** sit in
categories decided by a lab or vital, and Stage 7 currently misses **20** of them. The evidence
is already on disk and unused. For `patient_17774110`, the ground truth contains `E872` Acidosis
while its own labs record:

```
pH            7.11   (ref 7.35-7.45)
Bicarbonate   7.0    (ref 22-32)
Potassium     6.5    (ref 3.5-5.4)      -> hyperkalemia
Platelets    44.0    (ref 150-400)      -> thrombocytopenia
Creatinine    2.2    (ref 0.5-1.2)
```

This stage targets the pipeline's **coverage** gap, not the specificity gap -- an earlier test
showed specificity is largely unrecoverable here (near-misses need echo, ECG, laterality and
documentation detail, not lab values, and the pipeline is too-specific about as often as it is
too-vague).

## Two tiers

**Tier 1 -- BMI (`Z68.x`, `E66.x`).** Nearly deterministic: `Z68.37` *means* "BMI 37.0-37.9".
Notable because the pipeline currently predicts **zero** Z-chapter codes while 17.4% of the
ground truth is Z-chapter.

**Tier 2 -- threshold-based lab codes.** Electrolytes, acid-base, kidney, anemia,
thrombocytopenia, coagulation. Each rule fires off the lab's **own recorded reference range**
rather than a hardcoded cutoff, since ranges vary by assay and specimen.

## Two calibrations learned from the data, not assumed

**BMI must only fire at extremes.** Testing the raw mapping against ground truth: patients with
BMI 17.9, 15.8 and 42.1 all had `Z68` codes; patients at 21.3, 22.3, 24.4 and 25.2 had **none**.
Coders assign `Z68` when the BMI is clinically relevant, not routinely. Firing on every BMI
produces mostly false positives, so the rule is gated to `BMI < 20` or `BMI >= 35`.

**BMI is stale.** It comes from OMR outpatient records whose `charttime` can be years from the
admission. Two predictions landed one category off (36.7 -> `Z6836` where the coder recorded
`Z6837`; 31.7 -> `Z6831` vs `Z6832`). That is a data-recency limit, not a rule error, and it caps
what Tier 1 can achieve.

## Pipeline position

```
Stage 5   Ontology Routing Agent      symptoms  -> SNOMED concepts
Stage 6   Cross-Symptom Routing       concepts  -> clusters
Stage 7   Diagnosis Inference         clusters  -> named diagnoses
Stage 8   ICD-10 Mapping              diagnoses -> ICD codes
Stage 9   Prior-Admission Codes       history   -> ICD codes      (independent)
Stage 10  Lab/Vital Rules             labs      -> ICD codes      (independent)
Stage 11  Final Decision              combines 9 + 8 + 10
```
Run top to bottom. Stages 9 and 10 read only their own inputs, so they may run at any point
before Stage 11.


## 1. Setup

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"

PROJECT_ROOT    = NB_DIR.parent
RECORDS_DIR     = PROJECT_ROOT / "patient_records"
STAGE_6E_OUTPUT = "stage_06e_lab_vital_codes"

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Patients found: {len(patients)}")


Patients found: 15


## 2. Rules -- ADJUST HERE

Rules are chosen and retained on **measured per-rule precision**, not clinical plausibility.
From the first full run (53 firings, 16 correct, 30% overall):

| Rule | Fired | Correct | Precision | Status |
|---|---|---|---|---|
| `Z681` BMI <= 19.9 | 2 | 2 | 100% | keep |
| `E669` obesity | 3 | 2 | 67% | keep |
| `N179` acute kidney failure | 8 | 4 | 50% | keep |
| `Z6841` BMI 40-44.9 | 2 | 1 | 50% | keep |
| `E870` hypernatremia | 2 | 1 | 50% | keep |
| `E872` acidosis | 5 | 2 | 40% | keep |
| `E875` hyperkalemia | 3 | 1 | 33% | keep |
| `D696` thrombocytopenia | 6 | 1 | 17% | marginal |
| `D689` coagulation defect | 7 | 1 | 14% | marginal |
| `D649` anaemia | 12 | 1 | 8% | marginal |
| `E871` hyponatremia | 2 | 0 | 0% | marginal |

The three weak rules share a cause: low haemoglobin, deranged INR and low platelets are
near-universal in ICU patients, so they fire almost always and discriminate almost never.
They are kept behind `ENABLE_MARGINAL_RULES` rather than deleted -- they do add recall, and
whether that trade pays is a Stage 7 question. Turn the switch off to measure without them.

**New rules** target lab/vital-driven ground-truth codes 6e previously could not reach:

| Code | Trigger | Ground-truth instances |
|---|---|---|
| `I959` hypotension | lowest SBP < 90 mmHg | 2 |
| `E860` dehydration | BUN:creatinine ratio > 20 | 2 |
| `R791` abnormal coagulation profile | INR above reference | 1 |
| `J9601` acute respiratory failure with hypoxia | lowest SpO2 < 90% | 1 |

Vital rules use the **extreme** reading (min), not the last: a systolic that dipped to 78 is
codeable even if it recovered by the final observation. Coverage is the limit here -- SBP and
SpO2 are recorded for only 5 of 15 admissions, so these rules can fire at most 5 times.


In [2]:
ENABLE_TIER1 = True   # BMI -> Z68.x / E66.x
ENABLE_TIER2 = True   # threshold-based lab codes

# Rules whose measured precision was below 20% on this cohort. Kept behind a switch
# rather than deleted: they add recall, and whether that trade is worth it is a Stage 7
# question, not a Stage 6e one. Measured per-rule precision:
#     D696 thrombocytopenia 17% (1/6) | D689 coagulation 14% (1/7) | D649 anaemia 8% (1/12)
#     E871 hyponatremia 0% (0/2) | R791 coagulation profile 0% (0/7)
#     J9601 respiratory failure 0% (0/2) | I959 hypotension 0% (0/1)
# The lab-based ones fire on nearly every ICU patient, so they discriminate poorly. The
# vital-based ones (J9601, I959) failed for a different reason: SBP and SpO2 exist for only
# 5 of 15 admissions, and the patients carrying those codes are not the ones with the
# readings -- patient_17774110 has J9601 in its ground truth but a lowest SpO2 of 93%.
#
# Default is OFF: measured on the full cohort, history>=0.3 + strong rules only reaches
# macro F1 0.390, versus 0.375 with these included. They buy recall at more precision than
# they return.
ENABLE_MARGINAL_RULES = False

BMI_LOW_CUTOFF  = 20.0   # below this, code Z68.1
BMI_HIGH_CUTOFF = 35.0   # at/above this, code Z68.3x-Z68.45 (+ E66.9)

SBP_HYPOTENSION_CUTOFF = 90.0    # mmHg -- systolic below this
SPO2_HYPOXIA_CUTOFF    = 90.0    # % -- oxygen saturation below this
BUN_CR_RATIO_CUTOFF    = 20.0    # BUN:creatinine above this suggests pre-renal volume depletion

# (lab name, direction, icd_code, title, confidence, optional absolute cutoff, marginal?)
#   direction: "low" = below ref range / cutoff, "high" = above
LAB_RULES = [
    # --- measured >= 30% precision ---
    ("Potassium",        "high", "E875", "Hyperkalemia",                         0.85, None,  False),
    ("Potassium",        "low",  "E876", "Hypokalemia",                          0.85, None,  False),
    ("pH",               "low",  "E872", "Acidosis",                             0.85, 7.35,  False),
    ("pH",               "high", "E873", "Alkalosis",                            0.85, 7.45,  False),
    ("Sodium",           "high", "E870", "Hyperosmolality and hypernatremia",    0.75, None,  False),
    ("Creatinine",       "high", "N179", "Acute kidney failure, unspecified",    0.55, None,  False),
    # --- new rules, targeting lab/vital-driven ground-truth codes 6e did not reach ---
    ("INR(PT)",          "high", "R791", "Abnormal coagulation profile",         0.60, None,  True),
    # --- measured weak; behind ENABLE_MARGINAL_RULES ---
    ("Sodium",           "low",  "E871", "Hypo-osmolality and hyponatremia",     0.75, None,  True),
    ("Platelet Count",   "low",  "D696", "Thrombocytopenia, unspecified",        0.90, None,  True),
    ("Hemoglobin",       "low",  "D649", "Anemia, unspecified",                  0.80, None,  True),
    ("INR(PT)",          "high", "D689", "Coagulation defect, unspecified",      0.65, None,  True),
]


def _num(x):
    try:
        return float(str(x).strip())
    except (TypeError, ValueError):
        return None


def parse_ref_range(text):
    """'135.0-147.0' -> (135.0, 147.0); None when unparseable."""
    if not text:
        return None, None
    m = re.match(r"^\s*(-?\d+\.?\d*)\s*-\s*(-?\d+\.?\d*)\s*$", str(text))
    if not m:
        return None, None
    return float(m.group(1)), float(m.group(2))


def bmi_to_z68(bmi: float) -> str:
    """Z68 codes encode the BMI band directly: Z68.37 == 'BMI 37.0-37.9'."""
    if bmi < 20:  return "Z681"
    if bmi < 40:  return f"Z68{int(bmi)}"      # Z6820..Z6839
    if bmi < 45:  return "Z6841"
    if bmi < 50:  return "Z6842"
    if bmi < 60:  return "Z6843"
    if bmi < 70:  return "Z6844"
    return "Z6845"


def _vital_value(vitals: list, needle: str):
    """Most clinically relevant reading for a vital: the extreme, not the last.

    ICU vitals arrive as min/max/last summaries; a systolic that dipped to 78 matters
    for coding even if it recovered by the final reading. Falls back to `value` for
    OMR-sourced entries (BMI, weight), which carry no min/max.
    """
    for v in vitals:
        if needle.lower() in str(v.get("name", "")).lower():
            direct = _num(v.get("value"))
            if direct is not None:
                return direct
            return _num(v.get("min")) if _num(v.get("min")) is not None else _num(v.get("last"))
    return None


def codes_from_vitals(vitals: list) -> list:
    """Tier 1. BMI bands, plus vital-threshold codes."""
    if not ENABLE_TIER1:
        return []
    out = []

    bmi = _vital_value(vitals, "BMI")
    if bmi and bmi > 0 and not (BMI_LOW_CUTOFF <= bmi < BMI_HIGH_CUTOFF):
        # normal/overweight BMI is not routinely coded -- only the extremes are
        out.append({
            "icd_code": bmi_to_z68(bmi),
            "title": f"Body mass index [BMI] band for {bmi:.1f}",
            "confidence": 0.8,
            "evidence": f"BMI {bmi:.1f} (structured_vitals)",
            "tier": 1,
        })
        if bmi >= 30:
            out.append({"icd_code": "E669", "title": "Obesity, unspecified", "confidence": 0.7,
                        "evidence": f"BMI {bmi:.1f} >= 30", "tier": 1})

    sbp = _vital_value(vitals, "SBP")
    if ENABLE_MARGINAL_RULES and sbp is not None and 0 < sbp < SBP_HYPOTENSION_CUTOFF:
        out.append({"icd_code": "I959", "title": "Hypotension, unspecified", "confidence": 0.6,
                    "evidence": f"lowest SBP {sbp:.0f} < {SBP_HYPOTENSION_CUTOFF:.0f} mmHg", "tier": 1})

    spo2 = _vital_value(vitals, "SpO2")
    if ENABLE_MARGINAL_RULES and spo2 is not None and 0 < spo2 < SPO2_HYPOXIA_CUTOFF:
        out.append({"icd_code": "J9601", "title": "Acute respiratory failure with hypoxia",
                    "confidence": 0.5,
                    "evidence": f"lowest SpO2 {spo2:.0f}% < {SPO2_HYPOXIA_CUTOFF:.0f}%", "tier": 1})
    return out


def codes_from_labs(labs: list) -> list:
    """Tier 2. One code per triggered rule, keyed off each lab's own reference range."""
    if not ENABLE_TIER2:
        return []
    best = {}
    for l in labs:
        val = _num(l.get("value"))
        if val is None:
            continue
        name = l.get("name", "")
        lo, hi = parse_ref_range(l.get("ref_range"))
        prev = best.get(name)
        dev = abs(val - ((lo or 0) + (hi or 0)) / 2)
        if prev is None or dev > prev["dev"]:
            best[name] = {"val": val, "lo": lo, "hi": hi, "dev": dev, "unit": l.get("unit", "")}

    out = []
    for lab_name, direction, icd, title, conf, cutoff, marginal in LAB_RULES:
        if marginal and not ENABLE_MARGINAL_RULES:
            continue
        rec = best.get(lab_name)
        if not rec:
            continue
        val, lo, hi = rec["val"], rec["lo"], rec["hi"]
        if cutoff is not None:
            triggered = val < cutoff if direction == "low" else val > cutoff
            bound = cutoff
        else:
            if direction == "low" and lo is not None:
                triggered, bound = val < lo, lo
            elif direction == "high" and hi is not None:
                triggered, bound = val > hi, hi
            else:
                continue
        if triggered:
            out.append({"icd_code": icd, "title": title, "confidence": conf,
                        "evidence": f"{lab_name} {val}{rec['unit']} ({direction} vs {bound})",
                        "tier": 2, "marginal": marginal})

    # Derived rule: BUN:creatinine ratio, which no single threshold captures.
    bun, cr = best.get("Urea Nitrogen"), best.get("Creatinine")
    if bun and cr and cr["val"] > 0:
        ratio = bun["val"] / cr["val"]
        if ratio > BUN_CR_RATIO_CUTOFF:
            out.append({"icd_code": "E860", "title": "Dehydration", "confidence": 0.55,
                        "evidence": f"BUN:Cr ratio {ratio:.1f} > {BUN_CR_RATIO_CUTOFF:.0f}",
                        "tier": 2, "marginal": False})
    return out


## 3. Run across all admissions

In [3]:
all_results = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    adm_root = patient_dir / "admissions"
    for adm_dir in sorted(adm_root.iterdir()) if adm_root.exists() else []:
        labs_p  = adm_dir / "structured_labs.json"
        vitals_p = adm_dir / "structured_vitals.json"
        labs   = json.loads(labs_p.read_text(encoding="utf-8")) if labs_p.exists() else []
        vitals = json.loads(vitals_p.read_text(encoding="utf-8")) if vitals_p.exists() else []

        candidates = codes_from_vitals(vitals) + codes_from_labs(labs)
        # dedupe by code, keeping the strongest
        by_code = {}
        for c in candidates:
            if c["icd_code"] not in by_code or c["confidence"] > by_code[c["icd_code"]]["confidence"]:
                by_code[c["icd_code"]] = c
        ranked = sorted(by_code.values(), key=lambda c: (-c["confidence"], c["icd_code"]))

        output = {
            "patient_id": patient_id,
            "admission_id": adm_dir.name.replace("hadm_", ""),
            "tier1_enabled": ENABLE_TIER1,
            "tier2_enabled": ENABLE_TIER2,
            "n_candidates": len(ranked),
            "icd_candidates": ranked,
        }
        out_dir = adm_dir / STAGE_6E_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "lab_vital_codes.json", "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2)

        all_results.append(output)
        t1 = sum(1 for c in ranked if c["tier"] == 1)
        t2 = sum(1 for c in ranked if c["tier"] == 2)
        print(f'Patient {patient_id} | {adm_dir.name} | {len(ranked)} codes (tier1={t1}, tier2={t2})')

print(f"\nDone. {len(all_results)} admissions processed.")


Patient 10361982 | hadm_24286431 | 0 codes (tier1=0, tier2=0)
Patient 10426859 | hadm_29908281 | 1 codes (tier1=0, tier2=1)
Patient 10458324 | hadm_21744342 | 5 codes (tier1=1, tier2=4)
Patient 11251337 | hadm_29568708 | 2 codes (tier1=0, tier2=2)
Patient 11474876 | hadm_29672491 | 3 codes (tier1=0, tier2=3)
Patient 11607177 | hadm_23293838 | 6 codes (tier1=2, tier2=4)
Patient 12007928 | hadm_23749816 | 5 codes (tier1=2, tier2=3)
Patient 13196707 | hadm_21475988 | 5 codes (tier1=0, tier2=5)
Patient 13508515 | hadm_21834271 | 5 codes (tier1=2, tier2=3)
Patient 13952483 | hadm_23852410 | 5 codes (tier1=1, tier2=4)
Patient 16014068 | hadm_29042843 | 3 codes (tier1=0, tier2=3)
Patient 17774110 | hadm_27339772 | 7 codes (tier1=0, tier2=7)
Patient 18412100 | hadm_26093939 | 0 codes (tier1=0, tier2=0)
Patient 19104262 | hadm_24271247 | 4 codes (tier1=0, tier2=4)
Patient 19632936 | hadm_26696232 | 2 codes (tier1=0, tier2=2)

Done. 15 admissions processed.


## 4. Inspect one admission

In [4]:
EXAMPLE_IDX = 11
ex = all_results[EXAMPLE_IDX]
print(f'Patient {ex["patient_id"]} | Admission {ex["admission_id"]} | {ex["n_candidates"]} codes')
print()
for c in ex["icd_candidates"]:
    print(f'  T{c["tier"]}  {c["icd_code"]:<8} conf={c["confidence"]:.2f}  {c["title"][:40]:<42} {c["evidence"]}')


Patient 17774110 | Admission 27339772 | 7 codes

  T2  D696     conf=0.90  Thrombocytopenia, unspecified              Platelet Count 44.0K/uL (low vs 150.0)
  T2  E872     conf=0.85  Acidosis                                   pH 7.11units (low vs 7.35)
  T2  E875     conf=0.85  Hyperkalemia                               Potassium 6.5mEq/L (high vs 5.4)
  T2  D649     conf=0.80  Anemia, unspecified                        Hemoglobin 8.3g/dL (low vs 13.7)
  T2  E871     conf=0.75  Hypo-osmolality and hyponatremia           Sodium 129.0mEq/L (low vs 135.0)
  T2  D689     conf=0.65  Coagulation defect, unspecified            INR(PT) 3.4 (high vs 1.1)
  T2  N179     conf=0.55  Acute kidney failure, unspecified          Creatinine 2.2mg/dL (high vs 1.2)


## 5. Evaluate

Standalone quality of this source, plus each tier separately -- the point is to see whether it
earns a place in Stage 7's candidate pool, and which tier is carrying it.


In [6]:
def parse_ground_truth(adm_dir: Path) -> set:
    p = adm_dir / "ground_truth.txt"
    codes = set()
    if not p.exists():
        return codes
    for line in p.read_text(encoding="utf-8").splitlines():
        m = re.match(r"^\s*\d+\.\s+([A-Z0-9]+)\s+\u2014", line.strip())
        if m:
            codes.add(m.group(1).upper())
    return codes


def score(tier_filter=None):
    macro, tp_t, pr_t, gt_t = [], 0, 0, 0
    per_code = {"hit": [], "miss": []}
    for r in all_results:
        adm_dir = RECORDS_DIR / f'patient_{r["patient_id"]}' / "admissions" / f'hadm_{r["admission_id"]}'
        truth = parse_ground_truth(adm_dir)
        if not truth:
            continue
        pred = {c["icd_code"] for c in r["icd_candidates"]
                if tier_filter is None or c["tier"] == tier_filter}
        tp = len(pred & truth)
        p = tp / len(pred) if pred else 0.0
        rc = tp / len(truth)
        macro.append(2 * p * rc / (p + rc) if (p + rc) > 0 else 0.0)
        tp_t += tp; pr_t += len(pred); gt_t += len(truth)
        for c in pred:
            per_code["hit" if c in truth else "miss"].append((r["patient_id"], c))
    mp = tp_t / pr_t if pr_t else 0.0
    mr = tp_t / gt_t if gt_t else 0.0
    return {
        "macro_f1": round(sum(macro) / len(macro), 3) if macro else 0.0,
        "micro_f1": round(2 * mp * mr / (mp + mr), 3) if (mp + mr) > 0 else 0.0,
        "precision": round(mp, 3), "recall": round(mr, 3),
        "n_predicted": pr_t, "n_correct": tp_t,
    }, per_code


rows = []
for label, tf in [("Tier 1 only (BMI)", 1), ("Tier 2 only (labs)", 2), ("Both tiers", None)]:
    stats, _ = score(tf)
    rows.append({"source": label, **stats})
print(pd.DataFrame(rows).to_string(index=False))

print()
_, detail = score(None)
print(f'Correct ({len(detail["hit"])}):')
for pid, c in detail["hit"]:
    print(f'   patient {pid}: {c}')
print(f'\nIncorrect ({len(detail["miss"])}) -- first 15:')
for pid, c in detail["miss"][:15]:
    print(f'   patient {pid}: {c}')


# Per-rule precision -- the basis for keeping, dropping or gating each rule.
per_rule = {}
for r in all_results:
    adm_dir = RECORDS_DIR / f'patient_{r["patient_id"]}' / "admissions" / f'hadm_{r["admission_id"]}'
    truth = parse_ground_truth(adm_dir)
    if not truth:
        continue
    for c in r["icd_candidates"]:
        fired, correct = per_rule.get(c["icd_code"], (0, 0))
        per_rule[c["icd_code"]] = (fired + 1, correct + (1 if c["icd_code"] in truth else 0))

print()
print("Per-rule precision:")
print(f'{"code":<8}{"fired":>6}{"correct":>9}{"precision":>11}')
for code, (fired, correct) in sorted(per_rule.items(), key=lambda x: (-x[1][1] / x[1][0], -x[1][0])):
    print(f'{code:<8}{fired:>6}{correct:>9}{correct / fired:>10.0%}')


            source  macro_f1  micro_f1  precision  recall  n_predicted  n_correct
 Tier 1 only (BMI)     0.041     0.039      0.625   0.020            8          5
Tier 2 only (labs)     0.059     0.075      0.244   0.045           45         11
        Both tiers     0.091     0.107      0.302   0.065           53         16

Correct (16):
   patient 10426859: E872
   patient 10458324: Z681
   patient 11607177: N179
   patient 11607177: E669
   patient 12007928: N179
   patient 13196707: N179
   patient 13196707: E870
   patient 13508515: E669
   patient 13508515: Z6841
   patient 13952483: N179
   patient 13952483: Z681
   patient 13952483: D649
   patient 16014068: D696
   patient 17774110: E872
   patient 17774110: D689
   patient 19104262: E875

Incorrect (37) -- first 15:
   patient 10458324: E871
   patient 10458324: E875
   patient 10458324: E872
   patient 10458324: D649
   patient 11251337: N179
   patient 11251337: D649
   patient 11474876: D689
   patient 11474876: D649
   

## 6. Save summary

In [7]:
stats_both, _ = score(None)
stats_t1, _   = score(1)
stats_t2, _   = score(2)

summary = {
    "stage": "stage_06e_lab_vital_codes",
    "n_admissions": len(all_results),
    "parameters": {
        "enable_tier1": ENABLE_TIER1, "enable_tier2": ENABLE_TIER2,
        "bmi_low_cutoff": BMI_LOW_CUTOFF, "bmi_high_cutoff": BMI_HIGH_CUTOFF,
        "lab_rules": [{"lab": r[0], "direction": r[1], "icd": r[2]} for r in LAB_RULES],
    },
    "tier1": stats_t1, "tier2": stats_t2, "both": stats_both,
}
out_path = RECORDS_DIR / "stage_06e_summary.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {out_path}")
print(f'Both tiers -- macro F1 {stats_both["macro_f1"]}, precision {stats_both["precision"]}, '
      f'{stats_both["n_correct"]}/{stats_both["n_predicted"]} correct')


Saved: c:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_06e_summary.json
Both tiers -- macro F1 0.091, precision 0.302, 16/53 correct
